# TSI Multi-Scale Feature Pipeline
## Sensibilidad Temporal de Descriptores de Audio Artesanales

Este notebook implementa el pipeline experimental completo:
1. Carga de datos desde Google Drive
2. Extracción de features multi-escala (200ms, 2s, 5s)
3. Cómputo del Índice de Sensibilidad Temporal (TSI)
4. Comparación de 4 estrategias de fusión
5. Análisis de importancia y validación estadística

**Ejecución:** Google Colab Pro (GPU recomendada para MLP)

## 0. Setup & Configuration

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# Install dependencies
!pip install -q librosa scikit-learn torch torchaudio tqdm seaborn

# Clone the repo to get the src/ modules
import os
REPO_URL = "https://github.com/Gabrieleeh32159/my_paper.git"
REPO_DIR = "/content/my_paper"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull


In [ ]:

import os
import sys
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# === CONFIGURATION ===
# Code source (cloned repo)
REPO_DIR = Path('/content/my_paper')

# Google Drive paths for data persistence
DRIVE_ROOT = Path('/content/drive/MyDrive/tsi_experiments')
DATA_ROOT = DRIVE_ROOT / 'data'
FEATURES_ROOT = DRIVE_ROOT / 'features'  # Cache extracted features
RESULTS_ROOT = DRIVE_ROOT / 'results'    # Save experiment results

# Create output directories
FEATURES_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# Add repo's experiments/ to path so `from src.X import ...` works
sys.path.insert(0, str(REPO_DIR / 'experiments'))

# Dataset paths (data lives on Drive)
DATASET_PATHS = {
    'gtzan': DATA_ROOT / 'gtzan',
    'fma_small': DATA_ROOT,  # FMA expects fma_small/ and fma_metadata/ under root
    'mtat': DATA_ROOT / 'magnatagatune',
    'irmas': DATA_ROOT / 'irmas',
}

# Random seed for reproducibility
SEED = 42
np.random.seed(SEED)

print(f"Repo source: {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Features cache: {FEATURES_ROOT}")
print(f"Results output: {RESULTS_ROOT}")



In [ ]:
%cd {REPO_DIR}/experiments
import sys

# Añadimos el directorio raíz al path de Python
# Esto permite que 'import src' funcione desde cualquier subcarpeta
root_path = os.path.abspath(os.path.join(os.getcwd()))
print(root_path)
if root_path not in sys.path:
    sys.path.append(root_path)

In [ ]:

from src.features import FEATURE_DIMS, SCALES, TRACK_DIM
from src.data_loader import get_dataset
from src.fusion import early_fusion
from src.classifiers import get_classifier, RFClassifier
from src.tsi import compute_tsi, tsi_consistency
from src.evaluation import run_full_evaluation
from src.importance import aggregate_importance_by_descriptor_and_scale
from src.stats import spearman_with_bootstrap_ci, run_pairwise_comparisons

print("All modules loaded successfully.")
print(f"Feature dimensions per frame: {sum(FEATURE_DIMS.values())} ({FEATURE_DIMS})")
print(f"Track vector dim per scale: {TRACK_DIM}")
print(f"Temporal scales: {SCALES}")



## 1. Data Loading & Verification

In [ ]:
# Verify available datasets
print("=== Dataset Availability ===")
for name, path in DATASET_PATHS.items():
    exists = path.exists()
    print(f"  {name:12s}: {'✅' if exists else '❌'} {path}")
    if exists:
        # Count audio files
        audio_exts = {'.mp3', '.wav', '.au', '.ogg'}
        n_files = sum(1 for f in path.rglob('*') if f.suffix in audio_exts)
        print(f"               {n_files} audio files found")

In [ ]:

# === AUTO-DOWNLOAD DATASETS ===
# Downloads any missing dataset to Google Drive

import subprocess
import tarfile
import shutil


def download_gtzan(dest_root):
    """
    Download GTZAN dataset.
    Expected structure: dest_root/genres_original/{genre}/*.wav
    """
    dest_root = Path(dest_root)
    genres_dir = dest_root / 'genres_original'
    if genres_dir.exists() and any(genres_dir.iterdir()):
        print("  [gtzan] Already exists, skipping.")
        return True

    dest_root.mkdir(parents=True, exist_ok=True)
    tar_path = dest_root / 'genres.tar.gz'

    # Primary source
    url = "http://opihi.cs.uvic.ca/sound/genres.tar.gz"
    print(f"  [gtzan] Downloading from {url}...")

    !wget -q --show-progress -O {tar_path} {url} 2>&1 || true

    if not tar_path.exists() or tar_path.stat().st_size < 1000000:
        # Fallback: try Kaggle (requires kaggle to be configured)
        print("  [gtzan] Primary source failed. Trying Kaggle...")
        !pip install -q kagglehub 2>/dev/null
        try:
            import kagglehub
            kaggle_path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")
            kaggle_path = Path(kaggle_path)
            # Move genres_original to dest
            src_genres = kaggle_path / 'Data' / 'genres_original'
            if src_genres.exists():
                shutil.copytree(src_genres, genres_dir, dirs_exist_ok=True)
                print("  [gtzan] Downloaded via Kaggle.")
                return True
        except Exception as e:
            print(f"  [gtzan] Kaggle fallback failed: {e}")
            return False

    # Extract tar.gz
    if tar_path.exists() and tar_path.stat().st_size > 1000000:
        print("  [gtzan] Extracting...")
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(path=str(dest_root))
        tar_path.unlink()

        # Rename 'genres' -> 'genres_original' if needed
        if (dest_root / 'genres').exists() and not genres_dir.exists():
            (dest_root / 'genres').rename(genres_dir)
        print("  [gtzan] Done.")
        return True

    return False


def download_fma_small(dest_root):
    """
    Download FMA-small dataset (audio + metadata).
    Expected: dest_root/fma_small/ and dest_root/fma_metadata/
    """
    dest_root = Path(dest_root)
    audio_dir = dest_root / 'fma_small'
    metadata_dir = dest_root / 'fma_metadata'

    # Download audio if missing
    if not audio_dir.exists() or not any(audio_dir.iterdir()):
        dest_root.mkdir(parents=True, exist_ok=True)
        zip_path = dest_root / 'fma_small.zip'

        url = "https://os.unil.cloud.switch.ch/fma/fma_small.zip"
        print(f"  [fma_small] Downloading audio ({url})...")
        print("  [fma_small] This is ~7.2 GB, may take a while...")

        !wget -q --show-progress -O {zip_path} {url}

        if zip_path.exists() and zip_path.stat().st_size > 1000000:
            print("  [fma_small] Extracting audio...")
            !unzip -q -o {zip_path} -d {dest_root}
            zip_path.unlink()
            print("  [fma_small] Audio extracted.")
        else:
            print("  [fma_small] Audio download failed!")
            return False
    else:
        print("  [fma_small] Audio already exists.")

    # Download metadata if missing
    if not metadata_dir.exists() or not (metadata_dir / 'tracks.csv').exists():
        zip_path = dest_root / 'fma_metadata.zip'

        url = "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip"
        print(f"  [fma_small] Downloading metadata ({url})...")

        !wget -q --show-progress -O {zip_path} {url}

        if zip_path.exists() and zip_path.stat().st_size > 100000:
            print("  [fma_small] Extracting metadata...")
            !unzip -q -o {zip_path} -d {dest_root}
            zip_path.unlink()
            print("  [fma_small] Metadata extracted.")
        else:
            print("  [fma_small] Metadata download failed!")
            return False
    else:
        print("  [fma_small] Metadata already exists.")

    return True


def download_magnatagatune(dest_root):
    """
    Download MagnaTagATune dataset.
    Expected: dest_root/mp3/, dest_root/annotations_final.csv, dest_root/split/
    """
    dest_root = Path(dest_root)
    dest_root.mkdir(parents=True, exist_ok=True)

    # Download annotations
    annotations_file = dest_root / 'annotations_final.csv'
    if not annotations_file.exists():
        url = "https://mirg.city.ac.uk/datasets/magnatagatune/annotations_final.csv"
        print(f"  [mtat] Downloading annotations...")
        !wget -q --show-progress -O {annotations_file} {url}

    # Download audio (3 parts)
    mp3_dir = dest_root / 'mp3'
    if not mp3_dir.exists() or not any(mp3_dir.iterdir()):
        for part in range(1, 4):
            zip_name = f"mp3.zip.{part:03d}"
            zip_path = dest_root / zip_name
            url = f"https://mirg.city.ac.uk/datasets/magnatagatune/{zip_name}"
            print(f"  [mtat] Downloading audio part {part}/3...")
            !wget -q --show-progress -O {zip_path} {url}

        # Combine and extract
        print("  [mtat] Combining and extracting audio parts...")
        !cd {dest_root} && cat mp3.zip.001 mp3.zip.002 mp3.zip.003 > mp3_combined.zip
        !cd {dest_root} && unzip -q -o mp3_combined.zip

        # Cleanup
        for part in range(1, 4):
            (dest_root / f"mp3.zip.{part:03d}").unlink(missing_ok=True)
        (dest_root / 'mp3_combined.zip').unlink(missing_ok=True)
        print("  [mtat] Audio extracted.")
    else:
        print("  [mtat] Audio already exists.")

    # Download/create split files
    split_dir = dest_root / 'split'
    if not split_dir.exists():
        split_dir.mkdir(parents=True, exist_ok=True)
        print("  [mtat] Downloading split files...")

        # Use the keunwoochoi splits (standard in MIR literature)
        base_url = "https://raw.githubusercontent.com/keunwoochoi/magnatagatune-list/master"
        for split_file in ['train_list.txt', 'valid_list.txt', 'test_list.txt']:
            url = f"{base_url}/{split_file}"
            local_name = split_file.replace('_list', '')
            !wget -q -O {split_dir / local_name} {url}

        # The keunwoochoi files have format: "clip_path\tclip_id\n"
        # We need just clip_ids for our loader
        for fname in ['train.txt', 'valid.txt', 'test.txt']:
            fpath = split_dir / fname
            if fpath.exists():
                lines = fpath.read_text().strip().split('\n')
                # Extract clip_id (second column if tab-separated, else use as-is)
                clip_ids = []
                for line in lines:
                    parts = line.strip().split('\t')
                    if len(parts) >= 2:
                        clip_ids.append(parts[1])
                    else:
                        clip_ids.append(parts[0])
                fpath.write_text('\n'.join(clip_ids))
        print("  [mtat] Split files ready.")
    else:
        print("  [mtat] Split files already exist.")

    return True


def download_irmas(dest_root):
    """
    Download IRMAS dataset.
    Expected: dest_root/IRMAS-TrainingData/ and dest_root/IRMAS-TestingData-Part1/
    """
    dest_root = Path(dest_root)
    dest_root.mkdir(parents=True, exist_ok=True)

    train_dir = dest_root / 'IRMAS-TrainingData'
    test_dir = dest_root / 'IRMAS-TestingData-Part1'

    # Download training data
    if not train_dir.exists() or not any(train_dir.iterdir()):
        zip_path = dest_root / 'IRMAS-TrainingData.zip'
        url = "https://zenodo.org/record/1290750/files/IRMAS-TrainingData.zip"
        print(f"  [irmas] Downloading training data...")
        !wget -q --show-progress -O {zip_path} {url}

        if zip_path.exists() and zip_path.stat().st_size > 1000000:
            print("  [irmas] Extracting training data...")
            !unzip -q -o {zip_path} -d {dest_root}
            zip_path.unlink()
        else:
            print("  [irmas] Training data download failed!")
            return False
    else:
        print("  [irmas] Training data already exists.")

    # Download testing data
    if not test_dir.exists():
        zip_path = dest_root / 'IRMAS-TestingData-Part1.zip'
        url = "https://zenodo.org/record/1290750/files/IRMAS-TestingData-Part1.zip"
        print(f"  [irmas] Downloading testing data...")
        !wget -q --show-progress -O {zip_path} {url}

        if zip_path.exists() and zip_path.stat().st_size > 100000:
            print("  [irmas] Extracting testing data...")
            !unzip -q -o {zip_path} -d {dest_root}
            zip_path.unlink()
        else:
            print("  [irmas] Testing data download failed!")
            return False
    else:
        print("  [irmas] Testing data already exists.")

    return True


# === RUN DOWNLOADS ===
print("=" * 60)
print("CHECKING AND DOWNLOADING MISSING DATASETS")
print("=" * 60)

download_status = {}

# GTZAN
print("\n[1/4] GTZAN Dataset (~1.2 GB)")
gtzan_path = DATASET_PATHS['gtzan']
if not gtzan_path.exists() or not any(gtzan_path.iterdir()):
    download_status['gtzan'] = download_gtzan(gtzan_path)
else:
    print("  [gtzan] Already available.")
    download_status['gtzan'] = True

# FMA-small
print("\n[2/4] FMA-Small Dataset (~7.2 GB audio + metadata)")
fma_path = DATASET_PATHS['fma_small']
fma_audio = fma_path / 'fma_small'
fma_meta = fma_path / 'fma_metadata'
if not fma_audio.exists() or not fma_meta.exists():
    download_status['fma_small'] = download_fma_small(fma_path)
else:
    print("  [fma_small] Already available.")
    download_status['fma_small'] = True

# MagnaTagATune
print("\n[3/4] MagnaTagATune Dataset (~3.3 GB)")
mtat_path = DATASET_PATHS['mtat']
if not mtat_path.exists() or not (mtat_path / 'annotations_final.csv').exists():
    download_status['mtat'] = download_magnatagatune(mtat_path)
else:
    print("  [mtat] Already available.")
    download_status['mtat'] = True

# IRMAS
print("\n[4/4] IRMAS Dataset (~3.1 GB)")
irmas_path = DATASET_PATHS['irmas']
if not irmas_path.exists() or not (irmas_path / 'IRMAS-TrainingData').exists():
    download_status['irmas'] = download_irmas(irmas_path)
else:
    print("  [irmas] Already available.")
    download_status['irmas'] = True

# Summary
print("\n" + "=" * 60)
print("DOWNLOAD SUMMARY")
print("=" * 60)
for name, status in download_status.items():
    emoji = '✅' if status else '❌'
    print(f"  {name:12s}: {emoji}")


In [ ]:
# Load datasets (start with FMA-small since it's available)
# Uncomment datasets as they become available in Drive

datasets = {}

# FMA-small - should be available
try:
    datasets['fma_small'] = get_dataset('fma_small', str(DATASET_PATHS['fma_small']))
    print(f"FMA-small: {len(datasets['fma_small'])} tracks, {datasets['fma_small'].n_classes} classes")
except Exception as e:
    print(f"FMA-small failed: {e}")

# GTZAN
try:
    datasets['gtzan'] = get_dataset('gtzan', str(DATASET_PATHS['gtzan']))
    print(f"GTZAN: {len(datasets['gtzan'])} tracks, {datasets['gtzan'].n_classes} classes")
except Exception as e:
    print(f"GTZAN failed: {e}")

# MagnaTagATune
try:
    datasets['mtat'] = get_dataset('mtat', str(DATASET_PATHS['mtat']))
    print(f"MTAT: {len(datasets['mtat'])} clips, {datasets['mtat'].n_classes} tags")
except Exception as e:
    print(f"MTAT failed: {e}")

# IRMAS
try:
    datasets['irmas'] = get_dataset('irmas', str(DATASET_PATHS['irmas']))
    print(f"IRMAS: {len(datasets['irmas'])} fragments, {datasets['irmas'].n_classes} instruments")
except Exception as e:
    print(f"IRMAS failed: {e}")

print(f"\n=== Successfully loaded: {list(datasets.keys())} ===")

## 2. Multi-Scale Feature Extraction

Extract 7 descriptors × 3 scales for all tracks. Results are cached in Drive.

In [ ]:

import gc
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm

print("Feature extraction: sequential mode (in-kernel librosa)")


def extract_dataset_features(dataset_name, dataset, output_dir, sr=16000):
    """
    Sequential incremental feature extraction.
    - Loads existing cached features (skips already-done tracks)
    - Extracts ONLY missing tracks one by one with tqdm
    - Checkpoints feature arrays every 500 tracks to Drive
    - Labels/splits saved only at the end (lighter checkpoints)
    """
    from src.features import extract_multiscale_features, load_and_preprocess

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    n_tracks = len(dataset)

    # File paths
    short_path   = output_dir / f"{dataset_name}_short.npy"
    medium_path  = output_dir / f"{dataset_name}_medium.npy"
    long_path    = output_dir / f"{dataset_name}_long.npy"
    indices_path = output_dir / f"{dataset_name}_indices.npy"
    errors_path  = output_dir / f"{dataset_name}_errors.json"

    # ── Load existing cached data ─────────────────────────────────────────────
    done_set = set()
    cached_s, cached_m, cached_l, cached_idx = None, None, None, None

    if short_path.exists() and indices_path.exists():
        try:
            cached_s   = np.load(short_path)
            cached_m   = np.load(medium_path)
            cached_l   = np.load(long_path)
            cached_idx = np.load(indices_path)
            done_set   = set(cached_idx.tolist())
            print(f"  [{dataset_name}] Loaded {len(cached_idx)} cached tracks")
        except Exception as e:
            print(f"  [{dataset_name}] ⚠️ Cache corrupted, starting fresh: {e}")
            cached_s = cached_m = cached_l = cached_idx = None

    # Load previous errors
    all_errors = []
    if errors_path.exists():
        try:
            with open(errors_path) as f:
                all_errors = json.load(f)
        except Exception:
            pass
    error_indices = {e[0] for e in all_errors}

    # ── Determine missing tracks ──────────────────────────────────────────────
    missing = sorted(set(range(n_tracks)) - done_set - error_indices)
    print(f"  [{dataset_name}] Status: {len(done_set)} OK, {len(error_indices)} errors, "
          f"{len(missing)} remaining (total: {n_tracks})")

    if not missing:
        # Already complete — build final arrays
        if cached_s is not None:
            order = np.argsort(cached_idx)
            fi = cached_idx[order]
            fs, fm, fl = cached_s[order], cached_m[order], cached_l[order]
            labels = np.array([dataset.get_label(int(i)) for i in fi], dtype=object)
            splits = np.array([dataset.get_split(int(i)) for i in fi])
            np.save(output_dir / f"{dataset_name}_labels.npy", labels)
            np.save(output_dir / f"{dataset_name}_splits.npy", splits)
            print(f"  [{dataset_name}] ✅ Complete: {len(fi)} tracks × 192-d × 3 scales")
            return {'short': fs, 'medium': fm, 'long': fl}, labels, splits
        else:
            raise RuntimeError(f"No data found for {dataset_name}!")

    # ── Extract missing tracks sequentially ───────────────────────────────────
    new_s, new_m, new_l, new_idx = [], [], [], []
    new_errors = []
    checkpoint_every = 500
    gc_every = 50

    pbar = tqdm(missing, desc=f"[{dataset_name}]", unit="track")
    for i, track_idx in enumerate(pbar):
        filepath = str(dataset.get_audio_path(track_idx))
        try:
            y = load_and_preprocess(filepath, sr=sr)
            if len(y) < sr:
                new_errors.append([int(track_idx), filepath, "too short"])
                continue
            feats = extract_multiscale_features(y, sr=sr)
            new_s.append(feats['short'])
            new_m.append(feats['medium'])
            new_l.append(feats['long'])
            new_idx.append(int(track_idx))
            del y, feats
        except Exception as e:
            new_errors.append([int(track_idx), filepath, str(e)[:200]])
            continue

        # Periodic garbage collection
        if (i + 1) % gc_every == 0:
            gc.collect()

        # Checkpoint every N successfully extracted tracks
        if len(new_idx) > 0 and len(new_idx) % checkpoint_every == 0:
            pbar.set_postfix(saving="💾")
            _save_feature_checkpoint(
                dataset_name, output_dir,
                cached_s, cached_m, cached_l, cached_idx,
                new_s, new_m, new_l, new_idx,
                all_errors + new_errors
            )
            pbar.set_postfix(ok=len(done_set) + len(new_idx), err=len(all_errors) + len(new_errors))

    # ── Final save ────────────────────────────────────────────────────────────
    merged_errors = all_errors + new_errors
    seen = set()
    unique_errors = [e for e in merged_errors if e[0] not in seen and not seen.add(e[0])]

    # Merge cached + new feature arrays
    parts_s, parts_m, parts_l, parts_i = [], [], [], []
    if cached_s is not None:
        parts_s.append(cached_s); parts_m.append(cached_m)
        parts_l.append(cached_l); parts_i.append(cached_idx)
    if new_idx:
        parts_s.append(np.stack(new_s)); parts_m.append(np.stack(new_m))
        parts_l.append(np.stack(new_l)); parts_i.append(np.array(new_idx))

    if not parts_i:
        raise RuntimeError(f"No data for {dataset_name}!")

    fs = np.concatenate(parts_s); fm = np.concatenate(parts_m)
    fl = np.concatenate(parts_l); fi = np.concatenate(parts_i)

    # Deduplicate & sort by index
    _, upos = np.unique(fi, return_index=True)
    fi, fs, fm, fl = fi[upos], fs[upos], fm[upos], fl[upos]
    order = np.argsort(fi)
    fi, fs, fm, fl = fi[order], fs[order], fm[order], fl[order]

    # Compute labels and splits (only once at the end)
    labels = np.array([dataset.get_label(int(i)) for i in fi], dtype=object)
    splits = np.array([dataset.get_split(int(i)) for i in fi])

    # Save everything
    np.save(short_path,  fs)
    np.save(medium_path, fm)
    np.save(long_path,   fl)
    np.save(indices_path, fi)
    np.save(output_dir / f"{dataset_name}_labels.npy", labels)
    np.save(output_dir / f"{dataset_name}_splits.npy", splits)

    if unique_errors:
        with open(errors_path, 'w') as f:
            json.dump(unique_errors, f, indent=2, default=str)

    n_new = len(new_idx)
    print(f"  [{dataset_name}] ✅ {len(fi)} tracks (+{n_new} new), "
          f"{len(unique_errors)} errors × 192-d × 3 scales")

    # Free memory
    del new_s, new_m, new_l, parts_s, parts_m, parts_l
    del cached_s, cached_m, cached_l
    gc.collect()

    return {'short': fs, 'medium': fm, 'long': fl}, labels, splits


def _save_feature_checkpoint(dataset_name, output_dir,
                             cached_s, cached_m, cached_l, cached_idx,
                             new_s, new_m, new_l, new_idx, errors):
    """Lightweight checkpoint: save feature arrays + indices only (no labels)."""
    parts_s, parts_m, parts_l, parts_i = [], [], [], []
    if cached_s is not None:
        parts_s.append(cached_s); parts_m.append(cached_m)
        parts_l.append(cached_l); parts_i.append(cached_idx)
    if new_idx:
        parts_s.append(np.stack(new_s)); parts_m.append(np.stack(new_m))
        parts_l.append(np.stack(new_l)); parts_i.append(np.array(new_idx))

    if not parts_i:
        return

    fs = np.concatenate(parts_s); fm = np.concatenate(parts_m)
    fl = np.concatenate(parts_l); fi = np.concatenate(parts_i)

    _, upos = np.unique(fi, return_index=True)
    fi, fs, fm, fl = fi[upos], fs[upos], fm[upos], fl[upos]
    order = np.argsort(fi)
    fi, fs, fm, fl = fi[order], fs[order], fm[order], fl[order]

    np.save(output_dir / f"{dataset_name}_short.npy",   fs)
    np.save(output_dir / f"{dataset_name}_medium.npy",  fm)
    np.save(output_dir / f"{dataset_name}_long.npy",    fl)
    np.save(output_dir / f"{dataset_name}_indices.npy", fi)

    if errors:
        seen = set()
        unique = [e for e in errors if e[0] not in seen and not seen.add(e[0])]
        with open(output_dir / f"{dataset_name}_errors.json", 'w') as f:
            json.dump(unique, f, indent=2, default=str)

    del fs, fm, fl, fi, parts_s, parts_m, parts_l, parts_i
    gc.collect()



In [ ]:
# Extract features for all available datasets
all_features = {}
all_labels = {}
all_splits = {}

for name, ds in datasets.items():
    print(f"\nProcessing {name}...")
    feats, labels, splits = extract_dataset_features(name, ds, FEATURES_ROOT)
    all_features[name] = feats
    all_labels[name] = labels
    all_splits[name] = splits
    gc.collect()  # Liberar memoria entre datasets

print("\n=== Feature Extraction Complete ===")
for name in all_features:
    print(f"  {name}: {all_features[name]['short'].shape[0]} tracks")


## 3. TSI Computation

For each descriptor, train a classifier using ONLY that descriptor at each scale.
TSI(f) = (max_k Acc(f,k) - min_k Acc(f,k)) / Acc_chance

In [ ]:
def get_train_test_indices(splits, dataset_name):
    """Get train/test indices based on dataset splits."""
    if dataset_name == 'fma_small':
        train_idx = np.where((splits == 'train') | (splits == 'val'))[0]
        test_idx = np.where(splits == 'test')[0]
    elif dataset_name == 'mtat':
        train_idx = np.where(splits == 'train')[0]
        test_idx = np.where(splits == 'test')[0]
    elif dataset_name == 'irmas':
        train_idx = np.where(splits == 'train')[0]
        test_idx = np.where(splits == 'test')[0]
    else:  # gtzan - use first fold for initial TSI
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
        folds = list(skf.split(np.zeros(len(splits)), all_labels[dataset_name]))
        train_idx, test_idx = folds[0]
    return train_idx, test_idx

In [ ]:
# Compute TSI for all available datasets with Random Forest
tsi_results = {}  # {dataset_name: {tsi_scores, accuracy_matrix, optimal_scales}}

for ds_name in all_features.keys():
    if ds_name == 'mtat':
        # Skip multilabel for TSI (accuracy not well-defined)
        # Will handle separately with mAP-based TSI
        continue

    print(f"\n{'='*60}")
    print(f"Computing TSI for {ds_name} (n_classes={datasets[ds_name].n_classes})")
    print(f"{'='*60}")

    train_idx, test_idx = get_train_test_indices(all_splits[ds_name], ds_name)
    print(f"  Train: {len(train_idx)}, Test: {len(test_idx)}")

    tsi_scores, acc_matrix, opt_scales = compute_tsi(
        features=all_features[ds_name],
        labels=all_labels[ds_name],
        n_classes=datasets[ds_name].n_classes,
        classifier_name='rf',
        train_idx=train_idx,
        test_idx=test_idx,
    )

    tsi_results[ds_name] = {
        'tsi_scores': tsi_scores,
        'accuracy_matrix': acc_matrix,
        'optimal_scales': opt_scales,
    }

    # Print results
    print(f"\n  {'Descriptor':<20} {'TSI':>6} {'Short':>8} {'Medium':>8} {'Long':>8} {'Optimal':>8}")
    print(f"  {'-'*60}")
    for desc in FEATURE_DIMS.keys():
        tsi = tsi_scores[desc]
        acc_s = acc_matrix[desc]['short']
        acc_m = acc_matrix[desc]['medium']
        acc_l = acc_matrix[desc]['long']
        opt = opt_scales[desc]
        print(f"  {desc:<20} {tsi:>6.3f} {acc_s:>8.4f} {acc_m:>8.4f} {acc_l:>8.4f} {opt:>8}")

# Save TSI results
import json
tsi_save = {}
for ds_name, res in tsi_results.items():
    tsi_save[ds_name] = {
        'tsi_scores': res['tsi_scores'],
        'accuracy_matrix': res['accuracy_matrix'],
        'optimal_scales': res['optimal_scales'],
    }

with open(RESULTS_ROOT / 'tsi_results.json', 'w') as f:
    json.dump(tsi_save, f, indent=2)
print(f"\nTSI results saved to {RESULTS_ROOT / 'tsi_results.json'}")

## 4. Fusion Strategy Comparison

Compare 4 strategies: single-scale (short/medium/long), early fusion, late fusion, TSI-weighted fusion.

In [ ]:
def run_experiment_multiclass(ds_name, features, labels, splits, dataset, tsi_info, classifier_name='rf'):
    """
    Run full experiment for a multiclass dataset.
    Tests all 6 strategies with the given classifier.
    """
    train_idx, test_idx = get_train_test_indices(splits, ds_name)
    n_classes = dataset.n_classes
    strategies = ['short', 'medium', 'long', 'early', 'late', 'tsi_weighted']
    results = {}

    for strategy in strategies:
        print(f"    Strategy: {strategy}...", end=' ')

        # Get appropriate input dim
        if strategy in ('short', 'medium', 'long'):
            input_dim = 192
        elif strategy == 'early':
            input_dim = 576
        elif strategy == 'tsi_weighted':
            # Variable dim based on TSI selection
            input_dim = sum(4 * FEATURE_DIMS[d] for d in FEATURE_DIMS.keys())
        else:
            input_dim = 192

        clf = get_classifier(classifier_name, input_dim=input_dim, n_classes=n_classes)

        fusion_params = None
        if strategy == 'tsi_weighted' and tsi_info:
            fusion_params = {
                'tsi_scores': tsi_info['tsi_scores'],
                'optimal_scales': tsi_info['optimal_scales'],
            }

        result = run_full_evaluation(
            features=features,
            labels=labels,
            classifier=clf,
            train_idx=train_idx,
            test_idx=test_idx,
            strategy=strategy,
            task_type='multiclass',
            class_names=dataset.class_names,
            fusion_params=fusion_params,
        )
        results[strategy] = result
        print(f"Acc={result['accuracy']:.4f}, F1={result['f1_macro']:.4f}")

    return results

In [ ]:
# Run experiments for all datasets × classifiers
all_experiment_results = {}  # {dataset: {classifier: {strategy: metrics}}}
CLASSIFIERS = ['rf', 'svm', 'mlp']

for ds_name in all_features.keys():
    if datasets[ds_name].task_type == 'multilabel':
        continue  # Handle MTAT separately

    print(f"\n{'='*70}")
    print(f"DATASET: {ds_name} ({datasets[ds_name].n_classes} classes, {all_features[ds_name]['short'].shape[0]} tracks)")
    print(f"{'='*70}")

    all_experiment_results[ds_name] = {}
    tsi_info = tsi_results.get(ds_name)

    for clf_name in CLASSIFIERS:
        print(f"\n  Classifier: {clf_name.upper()}")
        print(f"  {'-'*50}")

        results = run_experiment_multiclass(
            ds_name=ds_name,
            features=all_features[ds_name],
            labels=all_labels[ds_name],
            splits=all_splits[ds_name],
            dataset=datasets[ds_name],
            tsi_info=tsi_info,
            classifier_name=clf_name,
        )
        all_experiment_results[ds_name][clf_name] = results

# Save results
results_save = {}
for ds_name, clf_results in all_experiment_results.items():
    results_save[ds_name] = {}
    for clf_name, strat_results in clf_results.items():
        results_save[ds_name][clf_name] = {}
        for strat, metrics in strat_results.items():
            results_save[ds_name][clf_name][strat] = {
                'accuracy': metrics['accuracy'],
                'f1_macro': metrics['f1_macro'],
            }

with open(RESULTS_ROOT / 'experiment_results.json', 'w') as f:
    json.dump(results_save, f, indent=2)
print(f"\nResults saved to {RESULTS_ROOT / 'experiment_results.json'}")

## 4b. MTAT (Multi-label) Experiment

In [ ]:
# MTAT experiment (if available)
if 'mtat' in all_features:
    print("\n" + "="*70)
    print("DATASET: MTAT (50 tags, multilabel)")
    print("="*70)

    features_mtat = all_features['mtat']
    labels_mtat = all_labels['mtat']
    splits_mtat = all_splits['mtat']

    train_idx = np.where(splits_mtat == 'train')[0]
    test_idx = np.where(splits_mtat == 'test')[0]

    # Compute class weights for MLP (inverse frequency)
    pos_counts = labels_mtat[train_idx].sum(axis=0)
    neg_counts = len(train_idx) - pos_counts
    class_weights = neg_counts / (pos_counts + 1e-6)

    mtat_results = {}
    strategies = ['short', 'medium', 'long', 'early', 'late']

    for strategy in strategies:
        print(f"  Strategy: {strategy}...", end=' ')

        if strategy in ('short', 'medium', 'long'):
            input_dim = 192
        else:
            input_dim = 576

        clf = get_classifier(
            'mlp', input_dim=input_dim, n_classes=50,
            task_type='multilabel', class_weights=class_weights
        )

        result = run_full_evaluation(
            features=features_mtat,
            labels=labels_mtat,
            classifier=clf,
            train_idx=train_idx,
            test_idx=test_idx,
            strategy=strategy,
            task_type='multilabel',
            class_names=datasets['mtat'].class_names,
        )
        mtat_results[strategy] = result
        print(f"mAP={result['mAP']:.4f}, AUC={result.get('roc_auc_macro', 'N/A')}")

    # Save MTAT results
    mtat_save = {s: {'mAP': r['mAP'], 'roc_auc_macro': r.get('roc_auc_macro')}
                 for s, r in mtat_results.items()}
    with open(RESULTS_ROOT / 'mtat_results.json', 'w') as f:
        json.dump(mtat_save, f, indent=2)
else:
    print("MTAT not available, skipping multilabel experiment.")

## 5. Feature Importance Analysis

In [ ]:
# Permutation Importance on early fusion with RF (primary analysis)
from sklearn.inspection import permutation_importance as sklearn_pi

importance_results = {}

for ds_name in all_features.keys():
    if datasets[ds_name].task_type == 'multilabel':
        continue

    print(f"\nComputing Permutation Importance for {ds_name}...")

    features = all_features[ds_name]
    labels = all_labels[ds_name]
    splits = all_splits[ds_name]

    train_idx, test_idx = get_train_test_indices(splits, ds_name)

    # Train RF on early fusion
    X_early = early_fusion(features)
    X_train, X_test = X_early[train_idx], X_early[test_idx]
    y_train, y_test = labels[train_idx], labels[test_idx]

    rf = RFClassifier()
    rf.fit(X_train, y_train)

    # Permutation importance (using underlying sklearn model + scaler)
    X_test_scaled = rf.scaler.transform(X_test)
    pi_result = sklearn_pi(
        rf.model, X_test_scaled, y_test,
        n_repeats=30, scoring='f1_macro',
        random_state=SEED, n_jobs=-1
    )

    # Aggregate by descriptor × scale
    imp_matrix = aggregate_importance_by_descriptor_and_scale(
        pi_result.importances_mean, vector_dim=192
    )

    importance_results[ds_name] = {
        'pi_mean': pi_result.importances_mean.tolist(),
        'pi_std': pi_result.importances_std.tolist(),
        'matrix': imp_matrix,
        'mdi': rf.feature_importances_.tolist(),
    }

    # Print importance matrix
    print(f"\n  Importance Matrix (PI, descriptor × scale):")
    print(f"  {'Descriptor':<20} {'Short':>10} {'Medium':>10} {'Long':>10}")
    print(f"  {'-'*52}")
    for desc in FEATURE_DIMS.keys():
        s = imp_matrix[desc]['short']
        m = imp_matrix[desc]['medium']
        l = imp_matrix[desc]['long']
        print(f"  {desc:<20} {s:>10.5f} {m:>10.5f} {l:>10.5f}")

# Save importance results
with open(RESULTS_ROOT / 'importance_results.json', 'w') as f:
    json.dump(importance_results, f, indent=2)
print(f"\nImportance results saved to {RESULTS_ROOT / 'importance_results.json'}")


## 6. Statistical Validation

In [ ]:
# Statistical tests: Wilcoxon + Cliff's delta
# Use per-fold accuracy (GTZAN) or per-class F1 (FMA/IRMAS) as paired samples

statistical_results = {}

for ds_name in all_experiment_results.keys():
    print(f"\n{'='*60}")
    print(f"Statistical Tests: {ds_name}")
    print(f"{'='*60}")

    # Use per-class F1 scores as paired observations
    clf_name = 'rf'  # Primary classifier
    strat_results = all_experiment_results[ds_name][clf_name]

    # Collect per-class F1 for each strategy
    paired_scores = {}
    for strategy, metrics in strat_results.items():
        if 'f1_per_class' in metrics:
            paired_scores[strategy] = np.array(metrics['f1_per_class'])

    if len(paired_scores) < 2:
        print("  Not enough strategies with per-class F1 for pairwise tests.")
        continue

    comparisons = run_pairwise_comparisons(paired_scores)
    statistical_results[ds_name] = comparisons

    print(f"\n  {'A vs B':<30} {'p-val':>8} {'Sig?':>6} {'Cliff δ':>8} {'Effect':>10}")
    print(f"  {'-'*65}")
    for comp in comparisons:
        pair = f"{comp['strategy_a']} vs {comp['strategy_b']}"
        sig = '✓' if comp['significant'] else '✗'
        print(f"  {pair:<30} {comp['p_value']:>8.5f} {sig:>6} {comp['cliffs_delta']:>8.3f} {comp['effect_magnitude']:>10}")

# Save statistical results
stats_save = {}
for ds_name, comps in statistical_results.items():
    stats_save[ds_name] = comps

with open(RESULTS_ROOT / 'statistical_tests.json', 'w') as f:
    json.dump(stats_save, f, indent=2, default=str)
print(f"\nStatistical results saved to {RESULTS_ROOT / 'statistical_tests.json'}")

In [ ]:
# TSI Consistency: Spearman correlation between datasets
print("\n=== TSI Consistency Across Datasets ===")

if len(tsi_results) >= 2:
    tsi_for_consistency = {
        ds_name: res['tsi_scores']
        for ds_name, res in tsi_results.items()
    }

    correlations = tsi_consistency(tsi_for_consistency)

    descriptors = list(FEATURE_DIMS.keys())
    configs = list(tsi_for_consistency.keys())

    for (cfg_a, cfg_b), rho in correlations.items():
        # Bootstrap CI
        x = np.array([tsi_for_consistency[cfg_a][d] for d in descriptors])
        y = np.array([tsi_for_consistency[cfg_b][d] for d in descriptors])
        ci_result = spearman_with_bootstrap_ci(x, y)

        consistent = '✓' if ci_result['consistent'] else '✗'
        print(f"  {cfg_a} vs {cfg_b}: ρ={ci_result['rho']:.3f} "
              f"[{ci_result['ci_lower']:.3f}, {ci_result['ci_upper']:.3f}] "
              f"Consistent(>0.7): {consistent}")
else:
    print("  Need at least 2 datasets for consistency analysis.")

## 7. Visualization & Results for Paper

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set IEEE paper style
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.figsize': (7, 4),
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

In [ ]:
# Figure 1: TSI Bar Chart per descriptor
fig, axes = plt.subplots(1, len(tsi_results), figsize=(4*len(tsi_results), 4), sharey=True)
if len(tsi_results) == 1:
    axes = [axes]

for ax, (ds_name, res) in zip(axes, tsi_results.items()):
    descriptors = list(res['tsi_scores'].keys())
    tsi_vals = [res['tsi_scores'][d] for d in descriptors]

    colors = ['#e74c3c' if v > 1.0 else '#3498db' if v > 0.5 else '#95a5a6' for v in tsi_vals]

    ax.barh(descriptors, tsi_vals, color=colors)
    ax.set_xlabel('TSI')
    ax.set_title(ds_name.upper())
    ax.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='High sensitivity')
    ax.axvline(x=0.5, color='blue', linestyle='--', alpha=0.5, label='Moderate')

axes[0].set_ylabel('Descriptor')
plt.suptitle('Temporal Sensitivity Index (TSI) per Descriptor', fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'fig_tsi_barchart.pdf')
plt.savefig(RESULTS_ROOT / 'fig_tsi_barchart.png')
plt.show()
print(f"Saved to {RESULTS_ROOT / 'fig_tsi_barchart.pdf'}")

In [ ]:
# Figure 2: 7×3 Sensitivity Matrix (Heatmap)
for ds_name, res in tsi_results.items():
    acc_matrix = res['accuracy_matrix']
    descriptors = list(FEATURE_DIMS.keys())
    scales = list(SCALES.keys())

    # Build matrix
    matrix = np.zeros((len(descriptors), len(scales)))
    for i, desc in enumerate(descriptors):
        for j, scale in enumerate(scales):
            matrix[i, j] = acc_matrix[desc][scale]

    fig, ax = plt.subplots(figsize=(5, 5))
    sns.heatmap(
        matrix, annot=True, fmt='.3f',
        xticklabels=['Short\n(200ms)', 'Medium\n(2s)', 'Long\n(5s)'],
        yticklabels=descriptors,
        cmap='YlOrRd', ax=ax,
        vmin=matrix.min() * 0.9,
        vmax=matrix.max() * 1.05,
    )
    ax.set_title(f'Accuracy Matrix (descriptor × scale) — {ds_name.upper()}')
    ax.set_xlabel('Temporal Scale')
    ax.set_ylabel('Descriptor')
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / f'fig_sensitivity_matrix_{ds_name}.pdf')
    plt.savefig(RESULTS_ROOT / f'fig_sensitivity_matrix_{ds_name}.png')
    plt.show()
    print(f"Saved: fig_sensitivity_matrix_{ds_name}.pdf")

In [ ]:
# Figure 3: Strategy Comparison Bar Chart
for ds_name, clf_results in all_experiment_results.items():
    strategies = ['short', 'medium', 'long', 'early', 'late', 'tsi_weighted']
    classifiers = list(clf_results.keys())

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(strategies))
    width = 0.25

    for i, clf_name in enumerate(classifiers):
        f1_scores = [clf_results[clf_name][s]['f1_macro'] for s in strategies]
        ax.bar(x + i*width, f1_scores, width, label=clf_name.upper())

    ax.set_xlabel('Fusion Strategy')
    ax.set_ylabel('F1 Macro')
    ax.set_title(f'Strategy Comparison — {ds_name.upper()}')
    ax.set_xticks(x + width)
    ax.set_xticklabels(strategies, rotation=20)
    ax.legend()
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / f'fig_strategy_comparison_{ds_name}.pdf')
    plt.savefig(RESULTS_ROOT / f'fig_strategy_comparison_{ds_name}.png')
    plt.show()

In [ ]:
# Figure 4: Importance Matrix Heatmap (early fusion, RF)
for ds_name, imp_res in importance_results.items():
    matrix = imp_res['matrix']
    descriptors = list(FEATURE_DIMS.keys())
    scales = list(SCALES.keys())

    mat_arr = np.zeros((len(descriptors), len(scales)))
    for i, desc in enumerate(descriptors):
        for j, scale in enumerate(scales):
            mat_arr[i, j] = matrix[desc][scale]

    fig, ax = plt.subplots(figsize=(5, 5))
    sns.heatmap(
        mat_arr, annot=True, fmt='.4f',
        xticklabels=['Short\n(200ms)', 'Medium\n(2s)', 'Long\n(5s)'],
        yticklabels=descriptors,
        cmap='viridis', ax=ax,
    )
    ax.set_title(f'Permutation Importance (descriptor × scale) — {ds_name.upper()}')
    ax.set_xlabel('Temporal Scale')
    ax.set_ylabel('Descriptor')
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / f'fig_importance_matrix_{ds_name}.pdf')
    plt.savefig(RESULTS_ROOT / f'fig_importance_matrix_{ds_name}.png')
    plt.show()

## 8. LaTeX Tables for Paper

In [ ]:
# Generate LaTeX table: Main results
def generate_results_latex(results, dataset_names):
    """Generate LaTeX table for paper."""
    strategies = ['short', 'medium', 'long', 'early', 'late', 'tsi_weighted']
    strat_labels = ['Short (200ms)', 'Medium (2s)', 'Long (5s)',
                    'Early Fusion', 'Late Fusion', 'TSI-Weighted']

    lines = []
    lines.append(r'\begin{table}[htp]')
    lines.append(r'\centering')
    lines.append(r'\caption{Comparison of fusion strategies (F1 Macro, RF classifier).}')
    lines.append(r'\label{tab:results_main}')

    cols = 'l' + 'c' * len(dataset_names)
    lines.append(r'\begin{tabular}{@{}' + cols + r'@{}}')
    lines.append(r'\toprule')

    header = r'\textbf{Strategy}'
    for ds in dataset_names:
        header += f' & \\textbf{{{ds.upper()}}}'
    header += r' \\'
    lines.append(header)
    lines.append(r'\midrule')

    for strat, label in zip(strategies, strat_labels):
        row = label
        for ds in dataset_names:
            if ds in results and 'rf' in results[ds] and strat in results[ds]['rf']:
                f1 = results[ds]['rf'][strat]['f1_macro']
                row += f' & {f1:.4f}'
            else:
                row += ' & ---'
        row += r' \\'
        lines.append(row)

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')

    return '\n'.join(lines)

# Generate and save
latex_table = generate_results_latex(all_experiment_results, list(all_experiment_results.keys()))
print(latex_table)

with open(RESULTS_ROOT / 'table_results_main.tex', 'w') as f:
    f.write(latex_table)
print(f"\nSaved to {RESULTS_ROOT / 'table_results_main.tex'}")

In [ ]:
# Generate LaTeX table: TSI values
def generate_tsi_latex(tsi_results):
    """Generate LaTeX table for TSI values."""
    descriptors = list(FEATURE_DIMS.keys())
    datasets_available = list(tsi_results.keys())

    lines = []
    lines.append(r'\begin{table}[htp]')
    lines.append(r'\centering')
    lines.append(r'\caption{Temporal Sensitivity Index (TSI) per descriptor and dataset.}')
    lines.append(r'\label{tab:tsi_values}')

    cols = 'l' + 'c' * len(datasets_available)
    lines.append(r'\begin{tabular}{@{}' + cols + r'@{}}')
    lines.append(r'\toprule')

    header = r'\textbf{Descriptor}'
    for ds in datasets_available:
        header += f' & \\textbf{{{ds.upper()}}}'
    header += r' \\'
    lines.append(header)
    lines.append(r'\midrule')

    for desc in descriptors:
        row = desc.replace('_', ' ').title()
        for ds in datasets_available:
            tsi = tsi_results[ds]['tsi_scores'][desc]
            row += f' & {tsi:.3f}'
        row += r' \\'
        lines.append(row)

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')

    return '\n'.join(lines)

latex_tsi = generate_tsi_latex(tsi_results)
print(latex_tsi)

with open(RESULTS_ROOT / 'table_tsi_values.tex', 'w') as f:
    f.write(latex_tsi)
print(f"\nSaved to {RESULTS_ROOT / 'table_tsi_values.tex'}")

## 9. Summary & Final Report

In [ ]:
# Print comprehensive summary
print("\n" + "="*80)
print("EXPERIMENT SUMMARY")
print("="*80)

print("\n--- TSI Rankings (Higher = More temporally sensitive) ---")
for ds_name, res in tsi_results.items():
    sorted_tsi = sorted(res['tsi_scores'].items(), key=lambda x: -x[1])
    print(f"\n  {ds_name.upper()}:")
    for i, (desc, tsi) in enumerate(sorted_tsi, 1):
        opt = res['optimal_scales'][desc]
        print(f"    {i}. {desc:<20} TSI={tsi:.3f}  (optimal: {opt})")

print("\n\n--- Best Strategy per Dataset (F1 Macro, RF) ---")
for ds_name, clf_results in all_experiment_results.items():
    rf_results = clf_results.get('rf', {})
    if rf_results:
        best = max(rf_results.items(), key=lambda x: x[1]['f1_macro'])
        print(f"  {ds_name.upper()}: {best[0]} (F1={best[1]['f1_macro']:.4f})")

print("\n\n--- Key Findings ---")
print("  1. Descriptors with HIGH temporal sensitivity (expect: MFCCs, ZCR):")
for ds_name, res in tsi_results.items():
    high_tsi = [(d, v) for d, v in res['tsi_scores'].items() if v > 0.5]
    if high_tsi:
        print(f"     {ds_name}: {', '.join(d for d, _ in high_tsi)}")

print("  2. Descriptors with LOW temporal sensitivity (expect: chroma, tonnetz):")
for ds_name, res in tsi_results.items():
    low_tsi = [(d, v) for d, v in res['tsi_scores'].items() if v < 0.3]
    if low_tsi:
        print(f"     {ds_name}: {', '.join(d for d, _ in low_tsi)}")

print("\n\n--- Files Saved to Drive ---")
for f in sorted(RESULTS_ROOT.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name} ({size_kb:.1f} KB)")

print("\n" + "="*80)
print("DONE. All results saved to:", RESULTS_ROOT)
print("="*80)